# 🎯 Module 02: LLM Fundamentals

> **Course:** Generative AI — From Scratch to GenAI Engineer  
> **Stage:** Stage 1 — Foundations  
> **Module:** 02 — LLM Fundamentals  
> **Planned Date:** July 4, 2026  
> **Level:** Beginner → Advanced  
> **Prerequisites:** Python, basic ML/DL, Module 01  
> **Previous:** 01 — GenAI Fundamentals  
> **Next:** 03 — LLM APIs and Model Providers

---

Now that we know **what** Generative AI is, it's time to understand the **models** that power it.

In this notebook we'll cover:

- What an **LLM** is
- How LLMs **generate text** (next-token prediction)
- **Tokens & tokenization**
- **Embeddings**
- **Transformer** architecture
- **Attention** (Q/K/V)
- Common **architecture families** (encoder-only, decoder-only, encoder-decoder)
- **Pretraining, post-training, inference**
- **Context windows**
- **Model parameters** vs **generation settings**
- **Temperature** and **Top-p**
- **Proprietary vs open-weight** models
- **Quantization** & local inference
- How to **choose** an LLM
- **Limitations**
- A **practical lab**

## 📚 Learning Objectives

By the end of this module you should be able to:

1. Explain **how an LLM generates text**
2. Understand **tokens & tokenization**
3. Explain **embeddings** and distinguish internal vs external embeddings
4. Explain the **Transformer** and **attention**
5. Distinguish **architecture families**
6. Explain **pretraining, post-training, and inference**
7. Understand **context windows**
8. Differentiate **model parameters** from **generation settings**
9. Explain **temperature** and **top-p** conceptually
10. Compare **hosted vs open-weight** models
11. Understand **quantization** and local inference
12. **Choose** a model for a task
13. Identify common **LLM limitations**
14. Complete the **practical lab**

---
## ⚙️ 0. Setup & Imports

In [ ]:
import math
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print('✅ Setup ready.')

---
## 🧠 1. What Is a Large Language Model?

A **Large Language Model (LLM)** is a machine learning model designed to **process and generate human language**.

Modern LLMs can also do:

- Code generation
- Summarization
- Classification
- Translation
- Question answering

Many modern LLMs are built using the **Transformer** architecture and trained on large collections of text.

### Example

**Input:**
```text
Explain an API in simple English.
```

**Possible output:**
```text
An API allows two software systems to communicate.
For example, a weather application can use an API
to request weather information from a server.
```

⚠️ An LLM is **not automatically** connected to the internet, your database, or your private documents — those require **integrations** and **permissions**.

---
## 🔤 2. How Does an LLM Generate Text?

Many LLMs generate text by **predicting the next token** given the tokens already present.

### Simplified process

```text
1. Receive input
2. Convert input into tokens
3. Convert tokens into numerical representations
4. Process through the model
5. Calculate scores for possible next tokens
6. Select a token per the generation strategy
7. Append to the sequence
8. Repeat until a stopping condition
```

### Example

Input: `"The sun rises in the"`

| Possible continuation | Illustrative probability |
| --- | ---: |
| east | 0.75 |
| morning | 0.10 |
| sky | 0.08 |
| west | 0.02 |
| Other | 0.05 |

> ⚠️ An LLM is **not** selecting complete sentences from a database — it generates sequences using learned parameters.

In [ ]:
# Illustrate next-token probabilities with a tiny example
tokens = ['east', 'morning', 'sky', 'west', 'other']
probs  = np.array([0.75, 0.10, 0.08, 0.02, 0.05])

print('Next-token probabilities (illustrative):')
for t, p in zip(tokens, probs):
    bar = '█' * int(p * 40)
    print(f'  {t:<10} {p:.2f}  {bar}')

print('\nThe model samples from this distribution → picks a token → repeats.')

---
## 🧩 3. Tokens and Tokenization

A **token** is a unit of text processed by a language model. Depending on the tokenizer, a token may be:

- A complete word
- Part of a word
- Punctuation
- A number or part of a number
- Whitespace or another unit

### Example

```text
I am learning Generative AI.
```

might become:

```text
["I", " am", " learning", " Generative", " AI", "."]
```

> This is illustrative only. Actual tokens depend on the tokenizer.

### Why Tokenization Matters

- **Input/output limits**
- **API cost** when pricing is token-based
- **Processing speed**
- How much text fits in a **context window**
- How different languages and formats are handled

> A 1,000-word document does **not** necessarily use the same number of tokens as another 1,000-word document.

In [ ]:
# Simple word-level tokenizer — real tokenizers (BPE, WordPiece) do more
def simple_tokenize(text):
    import re
    # Naive split: keep leading spaces, separate punctuation
    return re.findall(r'\s?\w+|[^\w\s]', text)

samples = [
    'I am learning Generative AI.',
    'def hello():\n    print("hi")',
    'مرحبا بالعالم',
]

for s in samples:
    toks = simple_tokenize(s)
    print(f'Text   : {s!r}')
    print(f'Tokens : {toks}')
    print(f'Count  : {len(toks)}')
    print()

print('Note: real LLM tokenizers (e.g., BPE) split words into subword units,')
print('so the token count differs from what a naive splitter produces.')

---
## 🧬 4. What Is an Embedding?

An **embedding** is a **numerical representation** of an item (token, sentence, document, image).

In language models, **token embeddings** represent tokens as vectors.

Other embedding models represent whole sentences or documents for **semantic search**.

Example vector:

```text
[0.21, -0.45, 0.73, 0.18, ...]
```

### Why Embeddings Matter

- Represent information numerically
- Enable **semantic search**
- Enable **similarity comparisons**
- Power **document retrieval**, **recommendations**, and **RAG**

### ⚠️ Important Distinction

| Internal token embeddings | External embedding model |
| --- | --- |
| Used *inside* the language model | Used *outside* to represent text |
| Fed to Transformer layers | Used for search / retrieval |
| Not typically interchangeable | Typically used directly |

We'll study practical embeddings & semantic search in **Module 06**.

---
## 🏛️ 5. Transformer Architecture

The **Transformer** is a major foundation of modern LLMs.

Earlier sequence models processed information **recurrently**. Transformers use **attention** to let representations incorporate information from **other relevant positions** in the input.

### Simplified pipeline

```text
Input Text
    |
    v
Tokenizer
    |
    v
Token IDs
    |
    v
Token Embeddings + Position Information
    |
    v
Transformer Layers
    |
    v
Output Scores (Logits)
    |
    v
Token Selection
    |
    v
Generated Text
```

### Main components

1. **Token embeddings** — token IDs → vectors
2. **Position information** — encodes order/position
3. **Attention** — combines information from relevant positions
4. **Feed-forward layers** — transform representations
5. **Normalization + residual connections** — stability & flow
6. **Output projection** — scores over possible next tokens

---
## 🎯 6. What Is Attention?

Attention determines **how information from different positions contributes** to a representation.

### Example

```text
The developer opened the project because it contained the required files.
```

When processing `it`, the model may use information from `the project`.

### Query, Key, Value

| | Meaning |
| --- | --- |
| **Query (Q)** | What a position is *looking for* |
| **Key (K)** | What each position *represents* |
| **Value (V)** | Information that can be *combined* |

### Scaled Dot-Product Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

For **decoder-only** models, **causal masking** prevents a position from attending to future tokens.

In [ ]:
# Tiny scaled dot-product attention demo
def attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / math.sqrt(d_k)
    weights = np.exp(scores - scores.max(axis=-1, keepdims=True))
    weights = weights / weights.sum(axis=-1, keepdims=True)
    return weights @ V, weights

np.random.seed(0)
T, d = 4, 8
Q = np.random.randn(T, d)
K = np.random.randn(T, d)
V = np.random.randn(T, d)

out, weights = attention(Q, K, V)

print('Attention weights (each row = query attends to keys):')
print(weights.round(3))
print('\nRow sums (should be 1.0):', weights.sum(axis=1).round(3))
print('Output shape:', out.shape)

In [ ]:
# Visualize attention weights
plt.figure(figsize=(5, 4))
plt.imshow(weights, cmap='viridis')
plt.colorbar(label='attention weight')
plt.title('Attention weights (query × key)')
plt.xlabel('Key position'); plt.ylabel('Query position')
plt.tight_layout(); plt.show()

---
## 🗂️ 7. Common LLM Architecture Families

| Family | Idea | Applications |
| --- | --- | --- |
| **Encoder-only** | Build contextual representations of the input | Classification, embeddings, extraction |
| **Decoder-only** | Generate sequences autoregressively | Chat, text generation, code |
| **Encoder-decoder** | Encode input, generate output | Translation, summarization, seq2seq |

These are **broad categories** — real systems may add components.

- Many chat-oriented LLMs use **decoder-only**
- Encoder-only models are useful for **classification/embeddings**
- Encoder-decoder shines on **translation/summarization**

> The best architecture depends on **task, model availability, performance, and constraints**.

---
## 🏋️ 8. Pretraining, Post-Training, Inference

### 8.1 Pretraining
Teaches the model **general patterns** from large datasets. For autoregressive LLMs, the common objective is **next-token prediction**.

### 8.2 Post-Training
Additional training / adaptation after pretraining:

- Supervised fine-tuning
- Instruction tuning
- Preference optimization
- Other alignment/task-specific techniques

### 8.3 Inference
Using a **trained** model to generate outputs.

> Sending a message to a provider's API usually triggers **inference** — it does **not** retrain the model.

### Comparison

| Concept | Purpose |
| --- | --- |
| Pretraining | Learn broad patterns |
| Post-training | Adapt behavior / capabilities |
| Inference | Generate outputs from trained model |
| Fine-tuning | Further train for a specific task (Module 13) |

---
## 🪟 9. Context Windows

A **context window** is the amount of **tokenized information** a model can process in a request.

### What counts toward it

- System instructions
- Conversation history
- Current user input
- Retrieved documents
- Tool results
- Other model-specific input

Output generation may also have separate limits or share the budget.

### Example

| Component | Tokens |
| --- | ---: |
| Instructions | 500 |
| Conversation history | 2,000 |
| Retrieved documents | 6,000 |
| Current question | 300 |
| **Total** | **8,800** |

### Engineering best practices

- Send **relevant** info — not everything
- Track input/output tokens
- Check the model's **current** limits
- Handle oversized requests gracefully
- Summarize/retrieve context when appropriate
- Test whether long contexts lose important info

> ⚠️ A larger context window does **not guarantee** the model will use every detail correctly.

---
## ⚖️ 10. Model Parameters vs Generation Settings

### Model Parameters
**Learned numerical values** inside the neural network. Adjusted during training.

### Generation Settings
Control **how a response is produced at inference time**:

- Temperature
- Top-p
- Max output tokens
- Stop sequences
- Provider-specific options

> These settings do **not** normally retrain the model.

| Model parameters | Generation settings |
| --- | --- |
| Learned during training | Chosen at inference |
| Part of the trained model | Control output generation |
| Change → retrain | Adjustable per request |

---
## 🌡️ 11. Temperature and Top-p

### Temperature
Adjusts the **distribution** used to select the next token.

- **Lower** → favors high-probability tokens
- **Higher** → allows lower-probability choices
- Does **not** guarantee factual correctness

### Top-p (nucleus sampling)
Restricts selection to a set of high-probability candidates whose **cumulative probability** reaches a threshold.

### Practical guidance

- Start with documented **defaults**
- Change **one** setting at a time
- Compare on **same** test prompts
- **Record** model, settings, results
- Follow provider docs

> ⚠️ The **same numeric value** does **not** mean equivalent behavior across different models.

In [ ]:
# Demonstrate how temperature sharpens/flattens a distribution
def softmax_with_temperature(logits, T):
    logits = np.asarray(logits)
    z = logits / T
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

logits = np.array([3.0, 2.0, 1.0, 0.5, 0.0])
tokens = ['east', 'morning', 'sky', 'west', 'other']

for T in [0.3, 1.0, 2.0]:
    probs = softmax_with_temperature(logits, T)
    print(f'Temperature = {T}')
    for tok, p in zip(tokens, probs):
        bar = '█' * int(p * 40)
        print(f'  {tok:<10} {p:.3f}  {bar}')
    print()

In [ ]:
# Demonstrate top-p (nucleus) truncation
def top_p_filter(probs, p):
    order = np.argsort(probs)[::-1]
    sorted_p = probs[order]
    cum = np.cumsum(sorted_p)
    keep = cum <= p
    if not keep.any():
        keep[0] = True
    kept = np.zeros_like(probs)
    for i, o in enumerate(order):
        kept[o] = sorted_p[i] if keep[i] else 0.0
    return kept / kept.sum()

probs = softmax_with_temperature(logits, 1.0)
print('Original distribution:', probs.round(3))
for p in [0.5, 0.9, 0.99]:
    filtered = top_p_filter(probs, p)
    print(f'Top-p = {p}: {filtered.round(3)}')

---
## 🆚 12. Proprietary vs Open-Weight Models

### Proprietary / Hosted
Accessed through a provider's API or hosted service.

**✅ Advantages:** easy setup, no infra, provider-managed.

**⚠️ Trade-offs:** usage costs, network dependency, provider-specific APIs, data-handling considerations.

### Open-Weight
Weights available under specified terms. Can run locally / on your infra.

**✅ Advantages:** more control, local/private inference, flexibility.

**⚠️ Trade-offs:** hardware, serving, maintenance, licensing obligations.

> ⚠️ "Open weights" does **not** automatically mean **unrestricted commercial use** or **fully open training data** — always check the license.

---
## 🧮 13. Quantization & Local Inference

### What is quantization?
Representing model values in **lower-precision** numerical formats.

- Reduces **memory use**
- Improves **inference efficiency**
- ⚠️ May affect **output quality**

### Local inference
Running a model on **your own** computer / infrastructure.

**Benefits:**
- More deployment control
- Reduced reliance on external services

**Trade-offs:**
- Hardware requirements
- Serving / maintenance work
- Licensing review

### Evaluate

- Available RAM / GPU memory
- Model size & quantization format
- Inference speed
- Output quality
- License & permitted use
- Privacy & logging

> A quantized model is **not necessarily** better than a full-precision model — choose based on **measured** quality and resource limits.

---
## 🎯 14. How to Choose an LLM

| Factor | Question |
| --- | --- |
| Task quality | Does it perform on your use case? |
| Accuracy | How often is it wrong? |
| Structured output | Can it produce the required format? |
| Tool calling | Does it support the required interface? |
| Context length | Fits input + output? |
| Latency | Fast enough? |
| Cost | Realistic cost? |
| Privacy | Meets data-handling needs? |
| Deployment | Runs in your environment? |
| License | Permitted uses? |
| Reliability | Service limits & failure modes? |

### Practical selection process

1. Define the task + success criteria
2. Prepare **representative** test examples
3. Pick a few candidate models
4. Run the **same** tests on each
5. Compare quality, latency, cost, failures
6. Check provider docs & licensing
7. Pick the **simplest** model that meets requirements
8. Re-evaluate when anything changes

> A smaller/cheaper model may be fine for classification/extraction; a bigger one may be justified for reasoning-heavy tasks.

---
## ⚠️ 15. Common LLM Limitations

### Hallucinations
Plausible-sounding but incorrect info.
**→** evaluation, trusted sources, retrieval, human review.

### Knowledge gaps
May not know recent/private info.
**→** approved retrieval / tools.

### Prompt sensitivity
Small prompt changes → different outputs.
**→** representative tests + regression tests.

### Long-context challenges
May overlook details in large inputs.
**→** improve context selection, evaluate long docs.

### Nondeterministic outputs
Same input → different responses.
**→** suitable settings, structured outputs, validation, tests.

### Cost & latency
Long inputs/outputs, repeated calls → higher cost & time.
**→** measure usage, reduce calls, choose appropriate models.

---
## 🧪 16. Practical Lab — Inspect LLM Behavior

### 🎯 Objective
Compare how a language model responds to **different prompts and settings**.

### 📋 Tasks

1. Choose a supported model (API or local)
2. Send a simple question
3. Record model ID + settings
4. Repeat the same task with a different prompt
5. Compare lower/higher temperature if supported
6. Inspect response length & consistency
7. **Check factual claims** — don't just judge fluency
8. Record token usage, latency, or estimated cost if available

### Suggested prompts

**Prompt A:**
```text
Explain the difference between a list and a tuple in Python.
Give one short example of each.
```

**Prompt B:**
```text
Explain the difference between a list and a tuple in Python.
Assume I am a beginner. Use a comparison table and one short example.
```

> Compare the outputs. Identify which changes came from the instructions vs from generation settings / model variability.

### 📦 Deliverables

- Runnable inference script / notebook
- Selected model identifier
- Prompts + supported settings tested
- Short comparison of outputs
- Notes on quality, limitations, latency, cost
- **No committed secrets**

**Completion rule:** mark complete only after running it and recording observations.

In [ ]:
# A minimal reproducible-evaluation scaffold you can reuse for any model
def make_experiment_log(model_id, prompt, settings, response, elapsed_s=None, tokens=None):
    return {
        'model_id': model_id,
        'prompt': prompt,
        'settings': settings,
        'response_len_chars': len(response),
        'response_len_words': len(response.split()),
        'elapsed_s': elapsed_s,
        'tokens': tokens,
    }

# Example scaffold (replace with real model call when ready)
log_entry = make_experiment_log(
    model_id='your-model-id',
    prompt='Explain list vs tuple in Python.',
    settings={'temperature': 0.7, 'top_p': 1.0, 'max_tokens': 200},
    response='A list is mutable; a tuple is immutable.',
    elapsed_s=0.42,
    tokens={'input': 12, 'output': 8},
)

print(json.dumps(log_entry, indent=2))

---
## 🎤 17. Interview Questions

**Q1. What is an LLM?**  
A model designed to process and generate language. Many use Transformer architectures, trained on large datasets.

**Q2. What is a token?**  
A unit of text processed by a model — a word, part of a word, punctuation, or another tokenizer-defined unit.

**Q3. What is the purpose of a Transformer?**  
It uses attention + other neural components to process relationships between tokens and produce useful representations.

**Q4. What is attention?**  
A mechanism that combines info from different positions based on learned relationships and attention rules.

**Q5. Difference between pretraining and inference?**  
Pretraining learns parameters from data. Inference uses a trained model to generate outputs.

**Q6. What is a context window?**  
The amount of tokenized information a model can process in a request.

**Q7. Model parameters vs temperature?**  
Parameters are learned values in the model. Temperature is an inference-time generation setting.

**Q8. Does a larger model always perform better?**  
No. Performance depends on model, training, task, evaluation, deployment.

**Q9. What is quantization?**  
Lower-precision representations to reduce resource use, possibly at the cost of some quality.

**Q10. Open-weight vs proprietary?**  
Open-weight: weights available under terms — more control. Proprietary: provider-controlled access. Availability & licenses vary.

**Q11. Why can an LLM hallucinate?**  
The generation process can produce plausible content not supported by facts or available evidence.

**Q12. How should you choose a model for production?**  
Test candidate models on representative tasks; compare quality, latency, cost, reliability, privacy, deployment, licensing.

---
## ✅ 18. Module Completion Checklist

- [ ] Explain how an LLM generates text
- [ ] Understand tokens & tokenization
- [ ] Explain embeddings (internal vs external)
- [ ] Understand Transformers & attention
- [ ] Distinguish architecture families
- [ ] Explain pretraining, post-training, inference
- [ ] Understand context windows
- [ ] Distinguish model params from generation settings
- [ ] Explain temperature & top-p
- [ ] Compare hosted vs open-weight models
- [ ] Understand quantization & local inference
- [ ] Complete the practical lab
- [ ] Answer the interview questions
- [ ] Commit code and notes to GitHub

---
## 🏁 19. Key Takeaways

- LLMs generate language using **learned parameters** + a **token-generation process**
- **Tokenization** determines how text becomes model input
- **Transformers** and **attention** are key foundations of modern LLMs
- **Pretraining, post-training, inference** serve different purposes
- **Context windows** constrain how much info can be processed
- **Model parameters ≠ generation settings**
- **Temperature & top-p** influence token selection — but don't guarantee quality
- **Hosted vs open-weight** models differ in ops & licensing
- **Quantization** reduces resources — but can affect quality
- Model selection = **evaluation + engineering requirements**, not popularity

### 🧭 Golden Rule

> **Understand the model, run an experiment, measure the result, document your findings, then move to the next module.**

---

### 🔗 What's Next?

**Module 03 — LLM APIs and Model Providers**

- Configure model-provider SDKs
- Manage API keys securely
- Send requests & handle responses
- Messages & conversation roles
- Streaming responses
- Timeouts, retries, backoff
- Rate limits & errors
- Async requests
- Token usage & cost tracking
- Provider-agnostic testing